# Forget-MI LoKU — Machine Unlearning Pipeline

Notebook thực hiện toàn bộ pipeline:

1. **Cell 1** — Kết nối Drive & Clone code từ GitHub
2. **Cell 2** — Giải nén data & models
3. **Cell 3** — Tiền xử lý (tạo all_data.tsv)
4. **Cell 4** — Huấn luyện LoKU Unlearning (với **auto-tracking**)
5. **Cell 5** — **Auto-commit & push** kết quả experiment lên GitHub

## Workflow cho mỗi experiment mới

1. Sửa `config.yaml` ở local → push code lên GitHub
2. Mở Colab → chạy lại Cell 1 (clone code mới)
3. **Sửa 2 biến** `EXP_NAME` và `HYPOTHESIS` ở đầu **Cell 4**
4. Chạy Cell 4 (10-60 phút) — auto tạo file MD + update INDEX
5. Chạy Cell 5 — auto push file MD về GitHub
6. (Local) `git pull` để lấy file MD về, điền 3 section: Observations / Conclusion / Next steps
7. `git push` — xong 1 experiment

> Lần đầu dùng Cell 5: setup Colab secrets `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME` (xem hướng dẫn trong cell).

In [1]:
# ====================================
# CELL 1: Kết nối Drive & Clone Code
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Clone Code từ GitHub
%cd /content
!rm -rf Forget-MI-LoKU
!git clone https://github.com/nhnhu146/Forget-MI-LoKU.git
%cd Forget-MI-LoKU

# 3. Cài đặt thư viện
!pip install -q pydicom scikit-image wandb pyyaml pandas
!pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")

Mounted at /content/drive
/content
Cloning into 'Forget-MI-LoKU'...
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 201 (delta 97), reused 150 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (201/201), 916.91 KiB | 4.13 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/Forget-MI-LoKU
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 100.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.5 MB/s eta 0:00:00:00:01
ERROR: p

In [2]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

🔍 Đang quét Drive để tìm file zip...
✅ Đã tìm thấy thư mục dự án tại: /content/drive/MyDrive/Forget-MI-Project

--- Bắt đầu giải nén ---
📦 Đang giải nén data.zip -> ./data
  🚚 Phát hiện lồng: data/data/ -> Đang gỡ...
  ✅ Gỡ lồng data/ thành công!
📦 Đang giải nén base_model.zip -> ./forgetme/training_original_model
  🚚 Phát hiện lồng: training_original_model/training_original_model/ -> Đang gỡ...
  ✅ Gỡ lồng training_original_model/ thành công!
📦 Đang giải nén retrained_model.zip -> ./model_retrained_3per
  🚚 Phát hiện lồng: model_retrained_3per/model_retrained_3per/ -> Đang gỡ...
  ✅ Gỡ lồng model_retrained_3per/ thành công!

--- Kiểm tra kết quả ---
✅ Text data: ./data/text_data
✅ Image data: ./data/img_data
✅ Base Model: ./forgetme/training_original_model/pytorch_model.bin
✅ Retrained Model: ./model_retrained_3per/pytorch_model.bin

🚀 Tất cả Dữ liệu & Model đã sẵn sàng!


In [3]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
import os
import shutil

# Tạo all_data.tsv từ các file báo cáo
!python make_tsv.py

# Xóa cache features cũ (nếu có)
!rm -f ./data/metadata/cachedfeatures_train_seqlen-*
!rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"): os.unlink("unlearning_output")
    else: shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output
print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")

📂 Sử dụng text_data tại: ./data/text_data
Đang đọc file split để lấy mapping study_id -> severity...
Đã nạp 12168 mapping study_id vào bộ nhớ.
Đang gộp file văn bản thành all_data.tsv...
  Đã quét 20000 file... (Khớp 548 báo cáo)
  Đã quét 40000 file... (Khớp 1242 báo cáo)
  Đã quét 60000 file... (Khớp 1720 báo cáo)
  Đã quét 80000 file... (Khớp 2255 báo cáo)
  Đã quét 100000 file... (Khớp 2848 báo cáo)
  Đã quét 120000 file... (Khớp 3370 báo cáo)
  Đã quét 140000 file... (Khớp 3854 báo cáo)
  Đã quét 160000 file... (Khớp 4323 báo cáo)
  Đã quét 180000 file... (Khớp 4836 báo cáo)
  Đã quét 200000 file... (Khớp 5346 báo cáo)
  Đã quét 220000 file... (Khớp 5920 báo cáo)
✅ HOÀN THÀNH! Đã trích xuất 6084 báo cáo trong 1.65 giây.
   File TSV: ./data/metadata/all_data.tsv

✅ Output sẽ được lưu tại: /content/drive/MyDrive/Forget-MI-Project/unlearning_output


In [ ]:
# ====================================
# CELL 3.5: Verify config — confirm code mới nhất từ GitHub
# ====================================
# Chạy cell này TRƯỚC Cell 4 để chắc chắn config đúng với exp đang định chạy.
# Nếu thấy giá trị CŨ → bạn quên push từ local, hãy push rồi rerun Cell 1.

print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "(forget_margin|eta_re_anchor|alpha|beta|theta|gamma|lora_r|lora_alpha|lora_target_modules|use_noise|unlearn_epochs|learning_rate):" -A 1 config.yaml | grep -v "^--"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config KHÔNG đúng với exp bạn định chạy:")
print("   1. Local: kiểm tra `git status` xem đã commit chưa")
print("   2. Local: `git push`")
print("   3. Colab: chạy lại Cell 1 (clone fresh)")
print("   4. Chạy lại Cell 3.5 này để verify")

In [ ]:
# ====================================
# CELL 4: Chạy LoKU Unlearning (với auto-tracking)
# ====================================
# ✏️  ĐỔI 2 BIẾN NÀY CHO MỖI EXPERIMENT MỚI:
EXP_NAME   = "exp03_classifier_unfrozen_neggrad"
HYPOTHESIS = "FIX BUG: unfreeze img_model.fc1 + text_model.classifier + them classification loss (CE retain, neg CE forget). Truoc do LoRA chi train BERT, eval dung img_logits nen ket qua bat bien. Doi: MIA giam, Forget AUC/F1 giam manh, Test giam nhe."

# (Optional) Đổi sang True nếu muốn xóa checkpoint cũ trước khi train
FRESH_START = True

# ----- Chạy training với auto-tracking -----
fresh_flag = "--fresh" if FRESH_START else ""
!PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py \
    --config config.yaml {fresh_flag} \
    --exp {EXP_NAME} \
    --hypothesis "{HYPOTHESIS}"

print("\n" + "="*60)
print(f"📄 Xem file experiment: experiments/exp_*_{EXP_NAME}.md")
print(f"📊 INDEX (bảng tổng):   experiments/INDEX.md")
print("="*60)
print("👉 Chạy CELL 5 để auto-commit & push lên GitHub")

In [5]:
# ====================================
# CELL 5: Auto-commit & push experiment results lên GitHub
# ====================================
# 3 cách setup credentials (chọn 1, theo độ tiện):
#
# CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ, setup 1 lần dùng mãi):
#   Tạo file /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json
#   với nội dung:
#   {
#     "GITHUB_TOKEN": "ghp_xxxxxxxxxxxx",
#     "GIT_EMAIL": "ban@gmail.com",
#     "GIT_NAME": "Nguyen Hoang Nhu"
#   }
#   Tạo token tại: https://github.com/settings/tokens (scope: repo)
#
# CÁCH B — Colab Secrets (CHỈ web colab.research.google.com):
#   Click 🔑 ở sidebar → Add secret: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#
# CÁCH C — Nhập tay mỗi session (lazy, không cần setup):
#   Bỏ qua A và B → cell sẽ tự hỏi token mỗi lần chạy
# ===========================================================
import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    # CÁCH A — file trên Drive
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Đã load credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')

    # CÁCH B — Colab Secrets (web Colab)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Đã load credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass

    # CÁCH B2 — environment variables
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Đã load credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))

    # CÁCH C — nhập tay (fallback)
    print("🔑 Nhập credentials thủ công (sẽ ẩn khi gõ token):")
    print("   (lần sau muốn auto, tạo file Drive theo CÁCH A ở comment trên)")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n


TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    # 1. Configure git identity (chỉ trong repo này, không ảnh hưởng global)
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"

    # 2. Inject token vào remote URL (chỉ trong session này)
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git

    # 3. Pull trước để tránh conflict
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5

    # 4. Stage CHỈ file experiment
    !git add experiments/ 2>/dev/null

    # 5. Hiển thị thay đổi
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")

        commit_msg = f"exp {EXP_NAME}: auto-tracked results"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}

        print(f"\n✅ Đã push lên GitHub")
        print(f"🔗 Xem online: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới để commit.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push. Setup theo CÁCH A/B/C ở comment trên.")


📦 Files sẽ commit:
   - experiments/INDEX.md
   - experiments/exp_002_forget_margin_20_no_anchor.md
[master ff5119d] exp forget_margin_20_no_anchor: auto-tracked results
 2 files changed, 260 insertions(+)
 create mode 100644 experiments/exp_002_forget_margin_20_no_anchor.md
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.09 KiB | 1.54 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/nhnhu146/Forget-MI-LoKU.git
   aa16ca3..ff5119d  master -> master

✅ Đã push lên GitHub
🔗 Xem online: https://github.com/nhnhu146/Forget-MI-LoKU/tree/master/experiments


In [6]:
!grep -B 1 -A 1 "forget_margin\|eta_re_anchor" config.yaml

    value: 0.5
  eta_re_anchor:
    value: 0.0                                            # [EXP 002] tắt re-anchor (was 0.5)
--
  # ===== Forget-margin (bounded loss) =====
  forget_margin:
    value: 20.0                                           # [EXP 002] tăng margin (was 8.0) — kích hoạt L_MD
